# 03 — Model: per-sector Ridge + ε-greedy contextual bandit

**Project:** Dynamic Retention Benchmarking (H03)

Two artefacts are persisted here:
1. A dictionary of `Ridge(alpha=1.0)` regressors keyed by sector — used by `/benchmark` to predict expected retention given a context.
2. An `EpsilonGreedyBandit` keyed by `(sector, action)` — used by `/benchmark` to surface a ranked top-N of HR levers.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from retention_bench.data import generate, write_processed, PROCESSED, ACTIONS
from retention_bench.features import NUMERIC_FEATURES, add_engineered
from retention_bench import models
from retention_bench.models import EpsilonGreedyBandit, train_per_sector_ridge, predict_retention

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

In [ ]:
PARQUET = PROCESSED / 'retention_panel.parquet'
df = pd.read_parquet(PARQUET) if PARQUET.exists() else generate()
df = add_engineered(df)
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(df))
n_train = int(0.8 * len(df))
df_train = df.iloc[perm[:n_train]].reset_index(drop=True)
df_test = df.iloc[perm[n_train:]].reset_index(drop=True)
df_train.shape, df_test.shape

## 1. Train per-sector Ridge regressors

In [ ]:
ridge_per_sector = train_per_sector_ridge(df_train, alpha=1.0)
y_pred = []
for _, row in df_test.iterrows():
    y_pred.append(predict_retention(ridge_per_sector, row.to_dict()))
y_pred = np.asarray(y_pred)
mae = mean_absolute_error(df_test['retention_rate'], y_pred)
r2 = r2_score(df_test['retention_rate'], y_pred)
print(f'held-out MAE: {mae:.4f}, R^2: {r2:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(df_test['retention_rate'], y_pred, alpha=0.4, s=14, color='#3a7ca5')
ax.plot([0.5, 1.0], [0.5, 1.0], color='red', ls='--')
ax.set_xlabel('observed retention'); ax.set_ylabel('predicted retention')
ax.set_title('Per-sector Ridge — predicted vs observed')
plt.tight_layout(); plt.show()

## 2. Per-sector MAE breakdown

In [ ]:
df_test = df_test.assign(pred=y_pred)
per_sector = (
    df_test.groupby('sector').apply(
        lambda x: pd.Series({
            'mae': mean_absolute_error(x['retention_rate'], x['pred']),
            'n': len(x),
        })
    ).round(4)
)
per_sector.sort_values('mae')

## 3. Coefficient inspection — per-sector slopes

Since each sector has its own Ridge, the slope vector tells us *what matters per sector*. Healthcare and Finance, for example, may pull harder on `manager_quality` than on `comp_percentile`.

In [ ]:
coef_rows = []
for sector, pipe in ridge_per_sector.items():
    ridge = pipe.named_steps['ridge']
    coef_rows.append({'sector': sector, **dict(zip(NUMERIC_FEATURES, ridge.coef_))})
coef_df = pd.DataFrame(coef_rows).set_index('sector').round(4)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(coef_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Per-sector Ridge slopes (scaled features)')
plt.tight_layout(); plt.show()

## 4. Train the ε-greedy contextual bandit

In [ ]:
bandit = EpsilonGreedyBandit(epsilon=0.10).fit(df_train)
ranks = []
for s in df_train['sector'].unique():
    for r in bandit.rank(s):
        ranks.append({'sector': s, **r})
rank_df = pd.DataFrame(ranks)
rank_df.head(10)

## 5. Best action per sector

The exploit-only ranking — the action with highest mean reward in each sector — is the operational primary output.

In [ ]:
best = (
    rank_df.sort_values(['sector', 'mean_reward'], ascending=[True, False])
    .drop_duplicates('sector')
    .reset_index(drop=True)
)
fig, ax = plt.subplots(figsize=(9, 3.6))
sns.barplot(data=best, x='sector', y='mean_reward', hue='action', dodge=False, ax=ax)
ax.set_title('Best action per sector (exploit-only mean reward)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()
best

## 6. Bandit cumulative regret simulation

We simulate the bandit's choice on a held-out fold and accumulate regret (best-arm reward minus chosen-arm reward) per row. Regret should plateau — the standard contextual-bandit convergence picture.

In [ ]:
best_arm_per_sector = best.set_index('sector')['mean_reward'].to_dict()
rng_b = np.random.default_rng(0)
regret = []
for _, row in df_test.iterrows():
    chosen = bandit.choose(row['sector'], rng=rng_b)
    chosen_mean = bandit.means.get((row['sector'], chosen), 0.0)
    regret.append(max(best_arm_per_sector.get(row['sector'], 0.0) - chosen_mean, 0.0))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(np.cumsum(regret), color='#9c6644')
ax.set_xlabel('held-out row index')
ax.set_ylabel('cumulative regret')
ax.set_title('ε-greedy cumulative regret on held-out')
plt.tight_layout(); plt.show()

## 7. Persist artefacts

In [ ]:
models.save(ridge_per_sector, 'ridge_per_sector.joblib')
models.save(bandit, 'bandit.joblib')
print('saved ridge_per_sector.joblib + bandit.joblib')

## 8. What this notebook commits to

These two artefacts are now the production-side answer to *'how does my retention rate compare to similar orgs in my sector, and which action should I pull next?'* The next step in `04_eval.ipynb` is to verify percentile-rank monotonicity, regret stability, and to generate the model-card numbers.